# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. `{'A': 2, 'B': 1}` — это «словарь-накопитель». Для нового ключа `.get(c, 0)` даёт 0, и мы прибавляем 1. Второй раз для `"A"` значение уже 1, станет 2.
2. `[0, 1, 1]` — список изменяемый и в функцию передаётся сам объект, а не копия. Два вызова `f(a)` дважды дописали `1` в тот же список.
3. `2` — множество хранит только уникальные значения. После `.lower()` `"Almaty"` и `"ALMATY"` совпадают, остаются `almaty` и `astana`.
4. `{'ab': 2, 'abc': 3}` — ключи в словаре уникальны. Второй `"ab"` просто перезаписывает первую запись (значение при этом то же самое, 2).
5. Сначала напечатается `5`, потом `None`. Внутри `g` стоит `print`, он только показывает значение. Явного `return` нет, поэтому функция возвращает `None`, и именно оно попадает в `x`.

In [ ]:
# Проверка предсказаний: запускаем фрагменты по очереди
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)

{'A': 2, 'B': 1}
[0, 1, 1]
2
{'ab': 2, 'abc': 3}
5
None


In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [ ]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


In [ ]:
# Что именно пришло из файла? Всё — строки.
print(rows[0])
print(type(rows[0]["amount"]))

{'order_id': '101674', 'order_date': '2026-09-01 08:00', 'customer_id': 'C00681', 'city': 'Алматы', 'category': 'Одежда', 'amount': '8070.0', 'delivery_days': '1.0', 'status': 'оплачен'}
<class 'str'>


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [ ]:
# --- Задача 1: to_num и clean_city ---
def to_num(s):
    """'15 990' или '8070.0' -> число (float). Пробелы внутри числа убираем."""
    return float(s.replace(" ", "").replace("\xa0", "").strip())

# Справочник: «как записано (в нижнем регистре, без пробелов по краям)» -> «как должно быть»
CITY_MAP = {
    "алматы": "Алматы",     "almaty": "Алматы",
    "астана": "Астана",     "astana": "Астана",
    "шымкент": "Шымкент",   "shymkent": "Шымкент",
    "караганда": "Караганда", "karaganda": "Караганда",
    "актобе": "Актобе",     "aktobe": "Актобе",
    "атырау": "Атырау",     "atyrau": "Атырау",
}

def clean_city(s):
    """' алматы', 'ALMATY', 'Almaty ' -> 'Алматы'. Неизвестное название -> KeyError (громко, а не молча)."""
    return CITY_MAP[s.strip().lower()]

print(to_num("15 990"), to_num("8070.0"))
print(clean_city(" алматы"), clean_city("Astana "), clean_city("SHYMKENT"))

# Чистим весь файл: создаём НОВЫЙ список словарей, а rows оставляем как есть (пригодится в задаче 9)
orders = [{**r, "amount": to_num(r["amount"]), "city": clean_city(r["city"])} for r in rows]

print("Было разных написаний города:", len({r["city"] for r in rows}))
print("Стало городов:", len({r["city"] for r in orders}), sorted({r["city"] for r in orders}))

15990.0 8070.0
Алматы Астана Шымкент
Было разных написаний города: 14
Стало городов: 6 ['Актобе', 'Алматы', 'Астана', 'Атырау', 'Караганда', 'Шымкент']


In [ ]:
# --- Задача 2: заказов в каждой категории (по убыванию) ---
by_cat = {}
for r in orders:
    by_cat[r["category"]] = by_cat.get(r["category"], 0) + 1

for cat, n in sorted(by_cat.items(), key=lambda t: t[1], reverse=True):
    print(f"{cat:<12}{n:>6}")

Одежда        1529
Электроника   1185
Дом           1105
Красота        991
Книги          653


In [ ]:
# --- Задача 3: выручка оплаченных и средний чек по категориям ---
cat_sum, cat_cnt = {}, {}
for r in orders:
    if r["status"] == "оплачен":
        cat = r["category"]
        cat_sum[cat] = cat_sum.get(cat, 0) + r["amount"]
        cat_cnt[cat] = cat_cnt.get(cat, 0) + 1

print(f"{'Категория':<12}{'Выручка, ₸':>16}{'Оплачено':>10}{'Ср. чек, ₸':>14}")
for cat in sorted(cat_sum, key=cat_sum.get, reverse=True):
    avg = cat_sum[cat] / cat_cnt[cat]
    print(f"{cat:<12}{cat_sum[cat]:>16,.0f}{cat_cnt[cat]:>10}{avg:>14,.0f}")

Категория         Выручка, ₸  Оплачено    Ср. чек, ₸
Электроника       65,270,240      1065        61,287
Одежда            18,321,080      1379        13,286
Дом               10,516,320       987        10,655
Красота            5,507,620       890         6,188
Книги              2,294,100       583         3,935


**Что видно в таблице.** Средний чек «Электроники» — 61 287 ₸, он выглядит подозрительно большим: у остальных категорий он в разы меньше. Причина — тестовый клиент `TEST` (см. задачу 7): три заказа по 9 999 990 ₸. Без них средний чек Электроники — около 33 210 ₸ (35 270 270 ₸ / 1 062 заказа).

In [ ]:
# --- Задача 4: заказов по месяцам ---
by_month = {}
for r in orders:
    m = r["order_date"][:7]
    by_month[m] = by_month.get(m, 0) + 1
print(by_month)

{'2026-09': 2782, '2026-10': 2681}


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [ ]:
# --- Задача 5: выручка оплаченных по месяцам и изменение окт/сен ---
def revenue_by_month(rs):
    """Выручка ОПЛАЧЕННЫХ заказов по месяцам. Принимает любой список заказов — пригодится в задаче 10."""
    res = {}
    for r in rs:
        if r["status"] == "оплачен":
            m = r["order_date"][:7]
            res[m] = res.get(m, 0) + r["amount"]
    return res

rev_m = revenue_by_month(orders)
for m, v in sorted(rev_m.items()):
    print(f"{m}: {v:>14,.0f} ₸")

change = (rev_m["2026-10"] / rev_m["2026-09"] - 1) * 100
print(f"Октябрь к сентябрю: {change:+.1f}%")

2026-09:     57,489,850 ₸
2026-10:     44,419,510 ₸
Октябрь к сентябрю: -22.7%


In [ ]:
# --- Задача 6: доля отменённых по городам ---
total, canceled = {}, {}
for r in orders:
    c = r["city"]
    total[c] = total.get(c, 0) + 1
    if r["status"] == "отменён":
        canceled[c] = canceled.get(c, 0) + 1

for c in sorted(total, key=lambda c: canceled.get(c, 0) / total[c], reverse=True):
    share = canceled.get(c, 0) / total[c] * 100      # .get(c, 0): вдруг в городе нет отмен
    print(f"{c:<10} {share:5.1f}%   ({canceled.get(c, 0)} из {total[c]})")

Шымкент     20.4%   (164 из 804)
Актобе      10.3%   (41 из 397)
Караганда    8.9%   (48 из 538)
Астана       8.8%   (123 из 1390)
Алматы       7.9%   (172 из 2188)
Атырау       7.5%   (11 из 146)


In [ ]:
# --- Задача 7: топ-5 клиентов по выручке оплаченных ---
cust_rev = {}
for r in orders:
    if r["status"] == "оплачен":
        cust_rev[r["customer_id"]] = cust_rev.get(r["customer_id"], 0) + r["amount"]

top5 = sorted(cust_rev.items(), key=lambda t: t[1], reverse=True)[:5]
for cid, v in top5:
    print(f"{cid:<8}{v:>14,.0f} ₸")

# Посмотрим на заказы лидера
for r in orders:
    if r["customer_id"] == top5[0][0]:
        print(r)

TEST        29,999,970 ₸
C01627         257,180 ₸
C00485         215,920 ₸
C01338         192,080 ₸
C02122         191,670 ₸
{'order_id': '105281', 'order_date': '2026-09-12 11:00', 'customer_id': 'TEST', 'city': 'Алматы', 'category': 'Электроника', 'amount': 9999990.0, 'delivery_days': '1.0', 'status': 'оплачен'}
{'order_id': '105282', 'order_date': '2026-09-25 15:00', 'customer_id': 'TEST', 'city': 'Алматы', 'category': 'Электроника', 'amount': 9999990.0, 'delivery_days': '1.0', 'status': 'оплачен'}
{'order_id': '105283', 'order_date': '2026-10-03 10:00', 'customer_id': 'TEST', 'city': 'Алматы', 'category': 'Электроника', 'amount': 9999990.0, 'delivery_days': '1.0', 'status': 'оплачен'}


**Задача 7.** На первом месте — клиент `TEST`: три оплаченных заказа по 9 999 990 ₸, всего 29 999 970 ₸. Это примерно в 117 раз больше, чем у второго места (257 180 ₸). Очевидно, это **тестовый аккаунт**, а не реальный покупатель: «магическая» сумма, одна категория (Электроника), один город. Он завышает выручку сентября и октября, средний чек электроники и ломает рейтинг клиентов. Такие записи надо исключать из отчётов.

In [ ]:
# --- Задача 8: уникальные клиенты по городам (словарь множеств) ---
cust_by_city = {}
for r in orders:
    cust_by_city.setdefault(r["city"], set()).add(r["customer_id"])

for c, s in sorted(cust_by_city.items(), key=lambda t: len(t[1]), reverse=True):
    print(f"{c:<10}{len(s):>6}")

Алматы      1351
Астана       981
Шымкент      655
Караганда    451
Актобе       358
Атырау       138


**Задача 8.** Множество (`set`) само отсекает повторы, поэтому `len(множество)` — это число *уникальных* клиентов. Если бы мы просто считали заказы по городам, клиент с пятью заказами считался бы пять раз. Заметьте: сумма по городам (3 934) больше, чем число уникальных клиентов во всём файле (2 003). Значит, один и тот же `customer_id` встречается в разных городах — в данных нет жёсткой привязки клиента к городу.

## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [ ]:
# --- Задача 9: полные дубли строк ---
seen = set()
dups = []
for r in rows:                      # берём «сырые» строки
    key = tuple(r.values())         # список нельзя положить в множество, а tuple — можно
    if key in seen:
        dups.append(r)              # такая строка уже была -> это дубль
    else:
        seen.add(key)

print("Дублей:", len(dups))
by_m = {}
for r in dups:
    by_m[r["order_date"][:7]] = by_m.get(r["order_date"][:7], 0) + 1
print("По месяцам:", by_m)

Дублей: 180
По месяцам: {'2026-09': 180}


In [ ]:
# --- Задача 10: выручка по месяцам без дублей и без тестового клиента ---
seen, unique_rows = set(), []
for r in orders:                    # те же приведённые заказы, что и в задачах 2–8
    key = tuple(r.values())
    if key not in seen:
        seen.add(key)
        unique_rows.append(r)

good = [r for r in unique_rows if r["customer_id"] != "TEST"]
print("Строк:", len(orders), "-> без дублей:", len(unique_rows), "-> без TEST:", len(good))

rev_good = revenue_by_month(good)   # та самая функция из задачи 5
for m, v in sorted(rev_good.items()):
    print(f"{m}: {v:>14,.0f} ₸")
change_good = (rev_good["2026-10"] / rev_good["2026-09"] - 1) * 100
print(f"Было: {change:+.1f}%   Стало: {change_good:+.1f}%")

Строк: 5463 -> без дублей: 5283 -> без TEST: 5280
2026-09:     35,243,410 ₸
2026-10:     34,419,520 ₸
Было: -22.7%   Стало: -2.3%


In [ ]:
# --- Задача 11: у кого больше всего заказов (после удаления дублей) ---
n_orders = {}
for r in good:
    n_orders[r["customer_id"]] = n_orders.get(r["customer_id"], 0) + 1

best = max(n_orders, key=n_orders.get)
print(best, n_orders[best])
# проверка на ничью: сколько клиентов с таким же максимумом?
print("Клиентов с максимумом:", sum(1 for v in n_orders.values() if v == n_orders[best]))

C01818 10
Клиентов с максимумом: 1


**Задача 12. Сообщение менеджеру:**

Выручка в октябре упала не на 22,7%, как показал первый расчёт, а всего на 2,3% (с 35,2 до 34,4 млн ₸). Первый расчёт был неверным, потому что в данных есть тестовый клиент `TEST` с заказами по 9 999 990 ₸ (20,0 млн ₸ в сентябре и 10,0 млн ₸ в октябре), а также 180 полных дублей строк, все из сентября (около 2,2 млн ₸ оплаченной выручки). Из-за них сентябрь оказался сильно завышен, и провал выглядел драматичнее, чем есть на самом деле. Рекомендую исключить тестовые заказы из отчётности и выяснить, почему в сентябрьской выгрузке появились дубли; отдельно стоит разобраться с долей отмен в Шымкенте (20,4% при 7,5–10,3% в других городах).